# 6. Análisis de predicción y causalidad

In [ ]:
# Preparar las variables necesarias para el modelo de regresión.
# Se asegura que las variables relevantes estén en formato numérico
# y se corrigen posibles valores perdidos o códigos especiales.

base_final['DOMINIO'] = pd.to_numeric(base_final['DOMINIO'], errors='coerce')
base_final['P207'] = pd.to_numeric(base_final['P207'], errors='coerce')
base_final['P208A'] = pd.to_numeric(base_final['P208A'], errors='coerce').replace(99, np.nan)
base_final['ESTRATO'] = pd.to_numeric(base_final['ESTRATO'], errors='coerce')
base_final['IDH'] = pd.to_numeric(base_final['IDH'], errors='coerce')
base_final['GASTO_BOLSILLO'] = pd.to_numeric(base_final['GASTO_BOLSILLO'], errors='coerce').fillna(0)
base_final['ANIOS_EDUC'] = pd.to_numeric(base_final['ANIOS_EDUC'], errors='coerce')

In [ ]:
# Crear variables derivadas necesarias para el análisis econométrico.
# Se construyen etiquetas del dominio geográfico, el logaritmo del gasto
# y variables dicotómicas para sexo y condición urbana.

diccionario_dominios = {
    1: "COSTA NORTE",
    2: "COSTA CENTRO",
    3: "COSTA SUR",
    4: "SIERRA NORTE",
    5: "SIERRA CENTRO",
    6: "SIERRA SUR",
    7: "SELVA",
    8: "LIMA METROPOLITANA"
}

base_final['DOMINIO_ETIQUETA'] = base_final['DOMINIO'].astype('Int64').map(diccionario_dominios)

# Log del gasto de bolsillo
base_final['log_gasto'] = np.log(base_final['GASTO_BOLSILLO'] + 1)

# Dummy de sexo: 1 si es mujer, 0 si es hombre
base_final['MUJER'] = np.where(base_final['P207'] == 2, 1, 0)

# Dummy de zona: 1 si reside en área urbana, 0 si reside en área rural
base_final['URBANO'] = np.where(base_final['ESTRATO'] <= 6, 1, 0)

In [ ]:
# Verificar que las variables del modelo estén listas antes de estimar la regresión
base_final[['log_gasto', 'ANIOS_EDUC', 'P208A', 'MUJER', 'URBANO', 'IDH', 'DOMINIO_ETIQUETA']].head()

,log_gasto,ANIOS_EDUC,P208A,MUJER,URBANO,IDH,DOMINIO_ETIQUETA
0,7.532624,14.0,74.0,0,1,0.741497,COSTA CENTRO
1,0.000000,11.0,70.0,1,1,0.741497,COSTA CENTRO
2,0.000000,14.0,22.0,1,1,0.741497,COSTA CENTRO
3,0.000000,11.0,19.0,0,1,0.741497,COSTA CENTRO
4,0.000000,11.0,48.0,0,1,0.741497,COSTA CENTRO


## 6.1. Modelo OLS

In [ ]:
# 1. Crear el diccionario de etiquetas para los códigos de DOMINIO
diccionario_dominios = {
    1: "COSTA NORTE",
    2: "COSTA CENTRO",
    3: "COSTA SUR",
    4: "SIERRA NORTE",
    5: "SIERRA CENTRO",
    6: "SIERRA SUR",
    7: "SELVA",
    8: "LIMA METROPOLITANA"
}

# 3. Crear la columna con nombres
base_final['DOMINIO_ETIQUETA'] = base_final['DOMINIO'].astype(int).map(diccionario_dominios)

# 4. Preparar las otras variables
# Log del gasto (sumamos 1 para manejar los ceros)
base_final['log_gasto'] = np.log(base_final['GASTO_BOLSILLO'] + 1)
# Dummy Mujer (P207: 1=Hombre, 2=Mujer)
base_final['MUJER'] = np.where(base_final['P207'] == 2, 1, 0)
# Dummy Urbano (Estrato 1-6)
base_final['URBANO'] = np.where(base_final['ESTRATO'] <= 6, 1, 0)

# 5. Correr la regresión usando LIMA METROPOLITANA como base
# Ahora P301B está limpio de valores "99"
formula_final = (
    'log_gasto ~ ANIOS_EDUC + P208A + MUJER + URBANO + IDH + ' # Corrected 'anios_educ' to 'ANIOS_EDUC'
    'C(DOMINIO_ETIQUETA, Treatment("LIMA METROPOLITANA"))'
)

m_final = smf.ols(formula_final, data=base_final).fit()

# 6. Ver el resumen
print(m_final.summary())

                            OLS Regression Results                            
Dep. Variable:              log_gasto   R-squared:                       0.045
Model:                            OLS   Adj. R-squared:                  0.045
Method:                 Least Squares   F-statistic:                     2160.
Date:                Tue, 07 Apr 2026   Prob (F-statistic):               0.00
Time:                        13:52:03   Log-Likelihood:            -1.3205e+06
No. Observations:              550443   AIC:                         2.641e+06
Df Residuals:                  550430   BIC:                         2.641e+06
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                                                            coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------

In [ ]:
print("=" * 65)
print("   RESUMEN FINAL — Determinantes del Gasto de Bolsillo en Salud")
print("=" * 65)
print(f"  Observaciones totales en base : {len(base_final):,}")
print(f"  Variables en el dataset       : {base_final.shape[1]}")
print(f"  Años cubiertos                : {sorted(base_final['AÑO'].unique())})")
print("-" * 65)
print("  Coeficientes clave (Modelo de Gasto):")
print(f"  β Años de eduación  = {m_final.params['ANIOS_EDUC']:.4f}    (p = {m_final.pvalues['ANIOS_EDUC']:.4f})")
print(f"  β Edad               = {m_final.params['P208A']:.4f}    (p = {m_final.pvalues['P208A']:.4f})")
print(f"  β Mujer              = {m_final.params['MUJER']:.4f}    (p = {m_final.pvalues['MUJER']:.4f})")
print(f"  β Urbano             = {m_final.params['URBANO']:.4f}    (p = {m_final.pvalues['URBANO']:.4f})")
print(f"  β IDH Provincial     = {m_final.params['IDH']:.4f}    (p = {m_final.pvalues['IDH']:.4f})")
print("-" * 65)
print(f"  R² (Bondad de ajuste) = {m_final.rsquared:.4f}")
print(f"  N (Muestra efectiva)  = {int(m_final.nobs):,}")
print("=" * 65)

   RESUMEN FINAL — Determinantes del Gasto de Bolsillo en Salud
  Observaciones totales en base : 606,619
  Variables en el dataset       : 148
  Años cubiertos                : [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)])
-----------------------------------------------------------------
  Coeficientes clave (Modelo de Gasto):
  β Años de eduación  = 0.0370    (p = 0.0000)
  β Edad               = 0.0114    (p = 0.0000)
  β Mujer              = 0.1969    (p = 0.0000)
  β Urbano             = 0.5694    (p = 0.0000)
  β IDH Provincial     = 2.7866    (p = 0.0000)
-----------------------------------------------------------------
  R² (Bondad de ajuste) = 0.0450
  N (Muestra efectiva)  = 550,443


## Análisis del modelo OLS


**Ecuación:**
$$\ln(Y) = \beta_0 + \beta_1 X + u$$

**Interpretación del coeficiente $\beta_1$:**
Por cada unidad adicional que aumenta la variable independiente ($X$), la variable dependiente ($Y$) cambia, en promedio, un **$(\beta_1 \times 100)\%$**.



* Educación ($\beta = 0.0425$): Encontramos un efecto positivo y significativo. Por cada año adicional en eduación, el gasto de bolsillo se incrementa en un 4.25%. Esto valida la hipótesis de que mayor capital humano no solo implica mayores ingresos, sino también una mayor demanda por servicios de salud de calidad o especializados que el sistema público no logra cubrir eficientemente.

* Edad ($\beta = 0.0135$): La edad actúa como un proxy del riesgo epidemiológico. Por cada año adicional, el gasto sube un 1.35%. Es un resultado esperado, pero confirma la presión que enfrentará una población en proceso de envejecimiento.

* Mujer($\beta = 0.2109$): Ser mujer incrementa el gasto de bolsillo en un 21.1% en comparación con los hombres, controlando por educación y zona.

* Urbano ($\beta = 0.6274$): Residir en una zona urbana se asocia con un gasto de bolsillo aproximadamente 62% mayor.


* IDH ($\beta = 2.78$): El coeficiente positivo del IDH resulta, a primera vista, paradójico, porque intuitivamente se esperaría que territorios con mayor desarrollo humano presenten una menor carga de gasto de bolsillo, al contar con mejores condiciones de vida y mayor disponibilidad de servicios. Sin embargo, el resultado sugiere que en provincias con mayor IDH también existe una mayor utilización de servicios de salud, una oferta más amplia —incluyendo servicios privados— y una mayor capacidad de pago, lo que puede traducirse en mayores desembolsos directos. En contraste, en territorios con menor IDH, un menor gasto de bolsillo no necesariamente refleja mayor protección financiera, sino también posibles barreras de acceso, subutilización de servicios o necesidades médicas no atendidas. Por ello, el hallazgo debe interpretarse como una asociación positiva entre desarrollo territorial y gasto observado, más que como evidencia de que el desarrollo humano empeora la protección financiera. En ese sentido, el resultado refleja una correlación empírica positiva, cuya interpretación requiere cautela y no debe asumirse automáticamente como una relación causal directa.